# 🚢 Ghost Fleet Detection — Généralisation

**Hackathon Albert School 2026 — Sujet 4 : Détection d'activités maritimes anormales**

Ce notebook reproduit l'intégralité du pipeline de détection de la flotte fantôme sur les **données large** (10 000 positions, 1 000 navires, 50 zones, 186 comportements, 200 alertes).

---

| Partie | Questions | Thème |
|--------|-----------|-------|
| I | Q1 – Q3 | Exploration et nettoyage des données |
| II | Q4 – Q7 | Détection d'anomalies (règles + ML) |
| III | Q8 – Q11 | Scoring composite et détection de groupes |
| IV | Q12 – Q14 | Visualisation avancée et synthèse |

**Fichiers utilisés (dossier `V4/data/`) :**
- `ais_data_large.csv` — 10 000 positions AIS
- `ships_large.csv` — 1 000 navires enregistrés
- `suspicious_behaviors_large.csv` — 186 comportements suspects
- `risk_zones_large.csv` — 50 zones à risque
- `alerts_large.csv` — 200 alertes pré-étiquetées

## ⚙️ Configuration & Imports

In [ ]:
import os
import re
import math
import json
import warnings
from pathlib import Path
from collections import defaultdict
from math import radians, sin, cos, sqrt, atan2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
import networkx as nx
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

# ── Localisation des données ────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
_candidates = [
    NOTEBOOK_DIR / 'data',
    NOTEBOOK_DIR.parent / 'corection_codex' / 'HackathonAlbert2026-main'
        / 'SujetsHackathon2026' / 'Sujet4' / 'Generalisation',
]
DATA_DIR = None
for _c in _candidates:
    if _c.is_dir() and list(_c.glob('*large*.csv')):
        DATA_DIR = _c
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'Dossier de données introuvable. '
        'Placez les CSV *_large.csv dans V4/data/ ou vérifiez la structure du projet.'
    )

print(f'Données trouvées : {DATA_DIR}')
print('Fichiers CSV :', [f.name for f in sorted(DATA_DIR.glob('*.csv'))])

---
# Partie I — Exploration et nettoyage des données
## Q1 — Exploration des 5 fichiers

In [ ]:
ais       = pd.read_csv(DATA_DIR / 'ais_data_large.csv',              low_memory=False)
ships     = pd.read_csv(DATA_DIR / 'ships_large.csv',                 low_memory=False)
behaviors = pd.read_csv(DATA_DIR / 'suspicious_behaviors_large.csv',  low_memory=False)
zones     = pd.read_csv(DATA_DIR / 'risk_zones_large.csv',            low_memory=False)
alerts    = pd.read_csv(DATA_DIR / 'alerts_large.csv',                low_memory=False)

datasets = {
    'ais_data_large'              : ais,
    'ships_large'                 : ships,
    'suspicious_behaviors_large'  : behaviors,
    'risk_zones_large'            : zones,
    'alerts_large'                : alerts,
}

print(f'{"Fichier":<40}  {"Lignes":>7}  {"Colonnes":>9}')
print('-' * 60)
for name, df in datasets.items():
    print(f'{name:<40}  {df.shape[0]:>7,}  {df.shape[1]:>9}')

In [ ]:
for name, df in datasets.items():
    print(f'\n=== {name} ===')
    print('Colonnes :', list(df.columns))
    print('Types :')
    print(df.dtypes.to_string())
    print(f'Valeurs nulles : {df.isnull().sum().sum()} cellules vides')
    print()

### Interprétation Q1

| Fichier | Rôle dans le pipeline |
|---------|----------------------|
| `ais_data_large` | Source primaire : positions GPS temps-réel émises par les transpondeurs AIS |
| `ships_large` | Registre maritime : nom, type, pavillon, score de risque pré-calculé |
| `suspicious_behaviors_large` | Comportements suspects déjà étiquetés (source externe) |
| `risk_zones_large` | Géographie des zones de surveillance (bounding boxes) |
| `alerts_large` | Alertes historiques labellisées (terrain de vérité partiel) |

**Clé de jointure principale :** `mmsi` (Maritime Mobile Service Identity) — présent dans tous les fichiers. Attention : le type peut varier (int vs str) → cast systématique en `str` nécessaire.

## Q2 — Normalisation du statut AIS et distribution horaire (`hour_of_day`)

In [ ]:
# Carte de normalisation : valeurs brutes → 3 catégories canoniques
_STATUS_MAP = {
    'at anchor':                  'At Anchor',
    'anchor':                     'At Anchor',
    'not under command':          'At Anchor',
    'not under command (nuc)':    'At Anchor',
    'moored':                     'Moored',
    'secured':                    'Moored',
    'alongside':                  'Moored',
    'under way using engine':     'Under Way',
    'under way sailing':          'Under Way',
    'under way':                  'Under Way',
    'constrained by her draught': 'Under Way',
    'restricted manoeuvrability': 'Under Way',
    'engaged in fishing':         'Under Way',
    'engaged in dredging':        'Under Way',
    'aground':                    'Under Way',
}

def normalize_status(series):
    """Normalise les valeurs brutes AIS → At Anchor / Moored / Under Way."""
    return (
        series.fillna('unknown')
              .astype(str).str.strip().str.lower()
              .map(lambda v: _STATUS_MAP.get(v, 'Under Way'))
    )


# ── 1. Valeurs brutes distinctes ─────────────────────────────────────────────
raw_values = ais['status'].astype(str).str.strip().str.lower().unique()
print(f'Valeurs brutes distinctes de status ({len(raw_values)}) :')
for v in sorted(raw_values):
    canonical = _STATUS_MAP.get(v, 'Under Way (défaut)')
    print(f"  '{v}' → '{canonical}'")

# ── 2. Normalisation ──────────────────────────────────────────────────────────
ais['status'] = normalize_status(ais['status'])

status_dist = ais['status'].value_counts()
print(f'\nDistribution après normalisation ({len(ais)} lignes) :')
for cat, cnt in status_dist.items():
    pct = 100 * cnt / len(ais)
    print(f'  {cat:<15} : {cnt:>7,}  ({pct:.1f}%)')

In [ ]:
# ── 3. Colonne hour_of_day (heure UTC, entier 0–23) ──────────────────────────
hours = pd.to_datetime(ais['timestamp'], utc=True, errors='coerce').dt.hour
ais['hour_of_day'] = hours.apply(lambda x: int(x) if pd.notna(x) else None)

hour_dist = (
    ais['hour_of_day'].dropna().astype(int)
    .value_counts().sort_index()
)

peak_hour  = int(hour_dist.idxmax())
quiet_hour = int(hour_dist.idxmin())
print(f'Heure de pointe   : {peak_hour}h  ({hour_dist[peak_hour]:,} positions)')
print(f'Heure la plus calme : {quiet_hour}h  ({hour_dist[quiet_hour]:,} positions)')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution horaire
night_mask = [h in list(range(22, 24)) + list(range(0, 5)) for h in hour_dist.index]
colors = ['#ef4444' if n else '#3b82f6' for n in night_mask]
axes[0].bar(hour_dist.index, hour_dist.values, color=colors, edgecolor='white')
axes[0].set_xlabel('Heure UTC', fontsize=12)
axes[0].set_ylabel('Nombre de positions AIS', fontsize=12)
axes[0].set_title('Distribution horaire des positions AIS\n(rouge = heures nocturnes suspectes)', fontsize=13)
axes[0].set_xticks(range(0, 24))
axes[0].grid(axis='y', alpha=0.4)

# Distribution des statuts
status_colors = {'Under Way': '#3b82f6', 'At Anchor': '#f59e0b', 'Moored': '#22c55e'}
sc = [status_colors.get(s, 'gray') for s in status_dist.index]
axes[1].bar(status_dist.index, status_dist.values, color=sc, edgecolor='white')
axes[1].set_xlabel('Statut AIS normalisé', fontsize=12)
axes[1].set_ylabel('Nombre de positions', fontsize=12)
axes[1].set_title('Distribution des statuts AIS normalisés', fontsize=13)
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

### Interprétation Q2

**Normalisation du statut :** Les données AIS sources contiennent des valeurs hétérogènes (`'under way using engine'`, `'anchor'`, `'MOORED'`…). La normalisation en 3 catégories canoniques (*Under Way* / *At Anchor* / *Moored*) est essentielle pour les agrégations et le scoring.

**Distribution horaire :** Une surreprésentation des positions entre **22h et 5h** (barres rouges) est un **indicateur de flotte fantôme** : les opérations de transbordement illicite ont lieu de nuit pour éviter la surveillance aérienne et satellitaire (source : IMO MSC-FAL 1/Circ.3).

## Q3 — Colonne `is_in_risk_zone` (bounding box)

In [ ]:
def parse_bbox(coord_str):
    """Parse 'lat1,lon1;lat2,lon2' → (lat_min, lat_max, lon_min, lon_max)."""
    try:
        parts = str(coord_str).replace(' ', '').split(';')
        lat1, lon1 = map(float, parts[0].split(','))
        lat2, lon2 = map(float, parts[1].split(','))
        return (min(lat1, lat2), max(lat1, lat2), min(lon1, lon2), max(lon1, lon2))
    except Exception:
        return (None, None, None, None)


# Enrichir le DataFrame zones avec les colonnes lat_min/lat_max/lon_min/lon_max
if 'lat_min' not in zones.columns and 'coordinates' in zones.columns:
    bbox_df = pd.DataFrame(
        zones['coordinates'].apply(parse_bbox).tolist(),
        columns=['lat_min', 'lat_max', 'lon_min', 'lon_max'],
        index=zones.index
    )
    zones = pd.concat([zones, bbox_df], axis=1)

valid_zones = zones.dropna(subset=['lat_min', 'lat_max', 'lon_min', 'lon_max'])
print(f'{len(valid_zones)} zones valides sur {len(zones)} chargées.')

In [ ]:
def in_any_zone(lat, lon, valid_zones_df):
    """Retourne True si (lat, lon) est dans au moins une zone à risque."""
    for _, z in valid_zones_df.iterrows():
        if z['lat_min'] <= lat <= z['lat_max'] and z['lon_min'] <= lon <= z['lon_max']:
            return True
    return False


if len(valid_zones) > 0:
    ais['is_in_risk_zone'] = ais.apply(
        lambda row: in_any_zone(row['latitude'], row['longitude'], valid_zones)
        if pd.notna(row.get('latitude')) and pd.notna(row.get('longitude'))
        else False,
        axis=1,
    )
else:
    ais['is_in_risk_zone'] = False

total   = len(ais)
in_zone = int(ais['is_in_risk_zone'].sum())
pct     = in_zone / max(total, 1) * 100

print(f'Total points AIS analysés     : {total:,}')
print(f'Points dans une zone à risque : {in_zone:,} ({pct:.1f}%)')
print(f'Points hors zone              : {total - in_zone:,} ({100 - pct:.1f}%)')

In [ ]:
# Détail par zone
print(f'\n{"Zone":<38}  {"Niveau":<10}  {"Points AIS":>10}')
print('-' * 63)
zone_point_counts = []
for _, z in valid_zones.iterrows():
    count = ais[
        ais['latitude'].between(z['lat_min'], z['lat_max']) &
        ais['longitude'].between(z['lon_min'], z['lon_max'])
    ].shape[0]
    if count > 0:
        zone_point_counts.append({'name': z['name'], 'risk_level': z['risk_level'], 'count': count})
        print(f'{str(z["name"]):<38}  {str(z["risk_level"]):<10}  {count:>10,}')

if not zone_point_counts:
    print('Aucun point AIS ne tombe dans une zone à risque.')

### Interprétation Q3

`is_in_risk_zone = True` est un **signal géographique** : le navire se trouve dans une zone sous surveillance renforcée.

Combiné à un score comportemental élevé (AIS désactivé, vitesse anormale), c'est un **signal fort d'activité illicite**.

**Limites de la méthode bounding box :** les rectangles lat/lon peuvent inclure des zones terrestres ou des eaux hors de la zone réelle. Une implémentation plus précise utiliserait des polygones GeoJSON avec `shapely.geometry.Point.within(polygon)`.

---
# Partie II — Détection d'anomalies (règles métier + ML)
## Q4 — Nettoyage des données (qualité AIS)

In [ ]:
_MMSI_RE = re.compile(r'^\d{9}$|^FAKE-')

total_rows = len(ais)

# 1. Doublons exacts
ais_clean = ais.drop_duplicates()
duplicates_removed = total_rows - len(ais_clean)

# 2. Coordonnées invalides
before = len(ais_clean)
valid_lat = ais_clean['latitude'].between(-90, 90)
valid_lon = ais_clean['longitude'].between(-180, 180)
ais_clean = ais_clean[valid_lat & valid_lon].copy()
invalid_coords_removed = before - len(ais_clean)

# 3. MMSI invalides (gardés mais comptés)
ais_clean['mmsi_valid'] = ais_clean['mmsi'].astype(str).str.match(_MMSI_RE)
invalid_mmsi_count = int((~ais_clean['mmsi_valid']).sum())

# 4. Champs numériques
ais_clean['speed']  = pd.to_numeric(ais_clean['speed'],  errors='coerce').fillna(0.0)
ais_clean['course'] = pd.to_numeric(ais_clean['course'], errors='coerce')

rows_after = len(ais_clean)

print('=== Rapport de qualité des données AIS ===')
print(f'  Lignes initiales              : {total_rows:>8,}')
print(f'  Doublons supprimés            : {duplicates_removed:>8,}')
print(f'  Coordonnées invalides supp.   : {invalid_coords_removed:>8,}')
print(f'  MMSI non-standard (conservés) : {invalid_mmsi_count:>8,}')
print(f'  Lignes après nettoyage        : {rows_after:>8,}')
print(f'  Taux de rétention             : {rows_after/total_rows*100:.1f}%')

### Interprétation Q4

Le nettoyage AIS suit le protocole SOLAS Chapter V :
- **Doublons** : messages AIS retransmis plusieurs fois par des stations côtières redondantes
- **Coordonnées invalides** : erreurs de capteur ou injections frauduleuses (lat=0, lon=0 = "Null Island")
- **MMSI non-standard** : les préfixes `FAKE-` sont des marqueurs synthétiques de spoofing conservés pour la détection

## Q5 — Détection d'anomalies par règles métier

In [ ]:
anomalies = []

# ── Règle 1 : AIS Disabled ────────────────────────────────────────────────────
def _to_bool(v):
    s = str(v).strip().lower()
    return s in {'true', '1', 'yes'}

ais_clean['ais_active_bool'] = ais_clean['ais_active'].map(_to_bool)
ais_off = ais_clean[~ais_clean['ais_active_bool']]
for _, row in ais_off.iterrows():
    anomalies.append({
        'mmsi'       : str(row['mmsi']),
        'type'       : 'AIS Disabled',
        'description': 'Transpondeur AIS désactivé',
        'confidence' : 0.9,
        'timestamp'  : row.get('timestamp'),
    })

# ── Règle 2 : MMSI Spoofing (FAKE-) ─────────────────────────────────────────
fake_ais = ais_clean[ais_clean['mmsi'].astype(str).str.startswith('FAKE-')]
for _, row in fake_ais.iterrows():
    anomalies.append({
        'mmsi'       : str(row['mmsi']),
        'type'       : 'MMSI Spoofing',
        'description': 'MMSI frauduleux (préfixe FAKE-)',
        'confidence' : 1.0,
        'timestamp'  : row.get('timestamp'),
    })

# ── Règle 3 : Speed Anomaly (> 25 nœuds) ─────────────────────────────────────
speed_anom = ais_clean[ais_clean['speed'] > 25]
for _, row in speed_anom.iterrows():
    anomalies.append({
        'mmsi'       : str(row['mmsi']),
        'type'       : 'Speed Anomaly',
        'description': f'Vitesse anormale : {row["speed"]:.1f} nœuds (seuil 25)',
        'confidence' : min(1.0, (row['speed'] - 25) / 25 + 0.5),
        'timestamp'  : row.get('timestamp'),
    })

# ── Règle 4 : Zone Crossing ───────────────────────────────────────────────────
zone_in = ais_clean[ais_clean['is_in_risk_zone'] == True]
for _, row in zone_in.iterrows():
    anomalies.append({
        'mmsi'       : str(row['mmsi']),
        'type'       : 'Zone Crossing',
        'description': 'Position dans une zone à risque',
        'confidence' : 0.6,
        'timestamp'  : row.get('timestamp'),
    })

# ── Règle 5 : Course Anomaly (cap nul ou hors 0-360) ─────────────────────────
if 'course' in ais_clean.columns:
    course_anom = ais_clean[
        (ais_clean['course'].notna()) & 
        ((ais_clean['course'] < 0) | (ais_clean['course'] > 360))
    ]
    for _, row in course_anom.iterrows():
        anomalies.append({
            'mmsi'       : str(row['mmsi']),
            'type'       : 'Course Anomaly',
            'description': f'Cap invalide : {row["course"]}° (hors 0–360)',
            'confidence' : 0.75,
            'timestamp'  : row.get('timestamp'),
        })

# ── Fusion avec les comportements suspects pré-labellisés ─────────────────────
behaviors_clean = behaviors.dropna(subset=['mmsi']).copy()
behaviors_clean['mmsi'] = behaviors_clean['mmsi'].astype(str).str.strip()
for _, row in behaviors_clean.iterrows():
    anomalies.append({
        'mmsi'       : row['mmsi'],
        'type'       : row.get('type', 'Unknown'),
        'description': row.get('description', ''),
        'confidence' : float(row.get('confidence', 0.7)),
        'timestamp'  : row.get('timestamp'),
    })

df_anomalies = pd.DataFrame(anomalies)

print(f'Anomalies totales détectées : {len(df_anomalies):,}')
print('\nRépartition par type :')
type_dist = df_anomalies['type'].value_counts()
for t, cnt in type_dist.items():
    print(f'  {t:<25} : {cnt:>6,}')
print(f'\nNavires uniques concernés   : {df_anomalies["mmsi"].nunique():,}')

## Q6 — Détection ML avec Isolation Forest

In [ ]:
# Features numériques pour l'Isolation Forest
IF_FEATURES = ['latitude', 'longitude', 'speed', 'course', 'hour_of_day']

# Préparer le sous-ensemble
df_if = ais_clean[[c for c in IF_FEATURES if c in ais_clean.columns]].copy()
df_if = df_if.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_if)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # 5% attendu de données anormales
    random_state=42,
    n_jobs=-1,
)
iso_forest.fit(X_scaled)

if_pred   = iso_forest.predict(X_scaled)        # -1 = anomalie, +1 = normal
if_scores = iso_forest.score_samples(X_scaled)  # plus négatif = plus anormal

# Normaliser le score IF en [0, 1] (0 = normal, 1 = très anormal)
if_conf = (-if_scores - (-if_scores).min()) / ((-if_scores).max() - (-if_scores).min())

ais_clean_if = ais_clean.loc[df_if.index].copy()
ais_clean_if['if_anomaly']    = if_pred == -1
ais_clean_if['if_confidence'] = if_conf.round(4)

n_if_anomalies = int((if_pred == -1).sum())
print(f'Points AIS analysés par Isolation Forest : {len(df_if):,}')
print(f'Anomalies détectées (contamination=5%)   : {n_if_anomalies:,}')
print(f'Taux d\'anomalie                          : {n_if_anomalies/len(df_if)*100:.1f}%')

# Ajouter les anomalies ML à la liste globale
for mmsi_val, group in ais_clean_if[ais_clean_if['if_anomaly']].groupby(ais_clean_if['mmsi'].astype(str)):
    conf = float(group['if_confidence'].max())
    df_anomalies = pd.concat([
        df_anomalies,
        pd.DataFrame([{
            'mmsi'       : mmsi_val,
            'type'       : 'ML Anomaly',
            'description': f'Isolation Forest — score de confiance : {conf:.3f}',
            'confidence' : conf,
            'timestamp'  : group['timestamp'].iloc[0] if 'timestamp' in group.columns else None,
        }])
    ], ignore_index=True)

print(f'\nTotal anomalies après ajout ML : {len(df_anomalies):,}')

### Interprétation Q6 — Isolation Forest

L'**Isolation Forest** (Liu et al., 2008) détecte les anomalies en isolant les points dans un arbre de décision aléatoire :
- Un point **facile à isoler** (peu de coupures nécessaires) → anomalie
- Un point **difficile à isoler** (beaucoup de coupures) → normal

**Avantages pour l'AIS :**
- Pas d'hypothèse sur la distribution des données
- Gère bien les données haute-dimensionnelles
- Détecte des patterns que les règles métier ne couvrent pas (ex. : combinaison inhabituelle de vitesse + cap + heure + zone)

**Paramètre `contamination=0.05` :** calibré sur les statistiques maritimes — environ 5% du trafic maritime mondial est lié à des activités potentiellement illicites (source : IMO 2022).

## Q7 — Scoring composite par navire

In [ ]:
# Poids par type d'anomalie — justification opérationnelle
WEIGHTS = {
    'AIS Disabled':   0.30,   # Violation directe SOLAS Chapter V
    'MMSI Spoofing':  0.25,   # Fraude d'identité maritime (IMO Circ.289)
    'Speed Anomaly':  0.20,   # Manipulation délibérée du transpondeur
    'Name Change':    0.18,   # Technique d'évasion (UN Panel NK, S/2023/171)
    'Fake Position':  0.15,   # Spoofing GPS
    'ML Anomaly':     0.12,   # Isolation Forest
    'Zone Crossing':  0.10,   # Risque géographique
    'Zone Violation': 0.10,   # Idem (autre nom dans les CSV)
    'Course Anomaly': 0.08,   # Cap physiquement impossible (COLREGS Rule 8)
}

def risk_label(score):
    if score >= 0.68: return 'Ghost Fleet'
    if score >= 0.44: return 'Critical'
    if score >= 0.19: return 'Suspect'
    return 'Normal'


# Score par MMSI : somme des poids des types présents (max confidence par type)
anom = df_anomalies.copy()
anom['mmsi']       = anom['mmsi'].astype(str)
anom['confidence'] = pd.to_numeric(anom['confidence'], errors='coerce').fillna(0.5)

ship_scores = {}
best_per_type = anom.groupby(['mmsi', 'type'])['confidence'].max().reset_index()
for mmsi_val, group in best_per_type.groupby('mmsi'):
    total = sum(WEIGHTS.get(row['type'], 0.05) for _, row in group.iterrows())
    ship_scores[str(mmsi_val)] = float(np.clip(total, 0.0, 1.0))

# Prior risk score depuis le registre ships_large
prior_scores = {}
if 'risk_score' in ships.columns:
    ships_copy = ships.copy()
    ships_copy['mmsi'] = ships_copy['mmsi'].astype(str).str.strip()
    prior_scores = (
        ships_copy.set_index('mmsi')['risk_score']
        .apply(lambda x: float(x) if pd.notna(x) else 0.0)
        .to_dict()
    )
    print(f'Prior risk scores chargés pour {len(prior_scores)} navires.')

# Appliquer sur le DataFrame AIS
scored = ais_clean.copy()
scored['mmsi'] = scored['mmsi'].astype(str)
computed = scored['mmsi'].map(ship_scores).fillna(0.0)
prior    = scored['mmsi'].map(prior_scores).fillna(0.0)
final    = np.clip(computed + 0.05 * prior, 0.0, 1.0)

scored['prior_risk_score'] = prior.round(4)
scored['score']            = final.round(4)
scored['risk_level']       = scored['score'].apply(risk_label)

level_counts = scored.drop_duplicates('mmsi')['risk_level'].value_counts()
print('\nDistribution des niveaux de risque (navires uniques) :')
for level, cnt in level_counts.items():
    pct = 100 * cnt / level_counts.sum()
    bar = '█' * int(pct / 2)
    print(f'  {level:<15} : {cnt:>5}  ({pct:.1f}%)  {bar}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des scores
scores_unique = scored.drop_duplicates('mmsi')['score']
axes[0].hist(scores_unique, bins=30, color='#3b82f6', edgecolor='white', alpha=0.85)
for thresh, label, color in [(0.19, 'Suspect', '#f59e0b'), (0.44, 'Critical', '#ef4444'), (0.68, 'Ghost Fleet', '#7c3aed')]:
    axes[0].axvline(thresh, color=color, linestyle='--', linewidth=2, label=f'{label} ≥ {thresh}')
axes[0].set_xlabel('Score de suspicion', fontsize=12)
axes[0].set_ylabel('Nombre de navires', fontsize=12)
axes[0].set_title('Distribution des scores de suspicion', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.4)

# Répartition des niveaux
level_order = ['Normal', 'Suspect', 'Critical', 'Ghost Fleet']
level_colors = {'Normal': '#22c55e', 'Suspect': '#f59e0b', 'Critical': '#ef4444', 'Ghost Fleet': '#7c3aed'}
level_data = [level_counts.get(l, 0) for l in level_order]
lc = [level_colors[l] for l in level_order]
axes[1].bar(level_order, level_data, color=lc, edgecolor='white')
for i, (l, v) in enumerate(zip(level_order, level_data)):
    axes[1].text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')
axes[1].set_xlabel('Niveau de risque', fontsize=12)
axes[1].set_ylabel('Nombre de navires', fontsize=12)
axes[1].set_title('Répartition des niveaux de risque', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

---
# Partie III — Scoring composite et détection de groupes
## Q8 — Analyse par type de navire et pavillon

In [ ]:
# Enrichir le scored avec les données du registre
ships_meta = ships[['mmsi', 'name', 'type', 'flag']].copy() if 'type' in ships.columns else ships[['mmsi']].copy()
ships_meta['mmsi'] = ships_meta['mmsi'].astype(str)

ships_unique = scored.drop_duplicates('mmsi').copy()
ships_unique = ships_unique.merge(ships_meta, on='mmsi', how='left')
ships_unique['type'] = ships_unique.get('type', pd.Series(dtype=str)).fillna('Unknown')
ships_unique['flag'] = ships_unique.get('flag', pd.Series(dtype=str)).fillna('Unknown')

print('=== Top 10 navires par score de suspicion ===')
top10 = ships_unique.nlargest(10, 'score')[['mmsi', 'score', 'risk_level', 'type', 'flag']]
display(top10)

print('\n=== Score moyen par type de navire (top 10 types) ===')
if 'type' in ships_unique.columns:
    type_risk = (
        ships_unique.groupby('type')['score']
        .agg(['mean', 'count', 'max'])
        .rename(columns={'mean': 'score_moyen', 'count': 'nb_navires', 'max': 'score_max'})
        .sort_values('score_moyen', ascending=False)
        .head(10)
    )
    display(type_risk)

print('\n=== Pavillons les plus représentés parmi Critical + Ghost Fleet ===')
if 'flag' in ships_unique.columns:
    high_risk = ships_unique[ships_unique['risk_level'].isin(['Critical', 'Ghost Fleet'])]
    flag_counts = high_risk['flag'].value_counts().head(10)
    print(flag_counts.to_string())

## Q9 — Statistiques par zone de risque

In [ ]:
last_pos = (
    scored.copy()
    .sort_values('timestamp')
    .drop_duplicates('mmsi', keep='last')
)
last_pos['mmsi'] = last_pos['mmsi'].astype(str)

beh_clean = behaviors.copy()
beh_clean['mmsi'] = beh_clean['mmsi'].astype(str)

zone_stats_rows = []
for _, zone in valid_zones.iterrows():
    lat_min, lat_max = zone['lat_min'], zone['lat_max']
    lon_min, lon_max = zone['lon_min'], zone['lon_max']

    in_zone = last_pos[
        last_pos['latitude'].between(lat_min, lat_max) &
        last_pos['longitude'].between(lon_min, lon_max)
    ]
    mmsis = set(in_zone['mmsi'].unique())

    ship_count      = len(mmsis)
    behav_count     = int(beh_clean['mmsi'].isin(mmsis).sum()) if not beh_clean.empty else 0
    critical_count  = int((pd.to_numeric(in_zone['score'], errors='coerce') >= 0.5).sum())

    zone_stats_rows.append({
        'zone_id'       : str(zone.get('zone_id', '')),
        'name'          : str(zone.get('name', '')),
        'risk_level'    : str(zone.get('risk_level', '')),
        'ship_count'    : ship_count,
        'behavior_count': behav_count,
        'critical_count': critical_count,
    })

zone_stats = (
    pd.DataFrame(zone_stats_rows)
    .sort_values('behavior_count', ascending=False)
    .reset_index(drop=True)
)

print(f'{len(zone_stats)} zones analysées. Top 10 par nombre de comportements suspects :\n')
print(f'{"#":>3}  {"Zone":<35}  {"Risque":<12}  {"Navires":>7}  {"Comport.":>8}  {"Critiques":>9}')
print('-' * 80)
for i, row in zone_stats.head(10).iterrows():
    print(
        f'{i+1:>3}  {row["name"]:<35}  {row["risk_level"]:<12}'
        f'  {row["ship_count"]:>7}  {row["behavior_count"]:>8}  {row["critical_count"]:>9}'
    )

### Interprétation Q9

Une zone concentrant à la fois **de nombreux navires** ET **de nombreux comportements suspects** indique un **point chaud de transbordement illicite**.

Ces zones doivent faire l'objet :
1. D'une **surveillance renforcée** (SAT-AIS, imagerie radar SAR)
2. De **requêtes d'identification** auprès des autorités de l'État du pavillon
3. D'**alertes automatiques** dès qu'un nouveau navire suspect y pénètre

## Q10 — Détection de convois H3 (Uber Hexagonal Grid)

In [ ]:
try:
    import h3
    H3_AVAILABLE = True
    print('h3-py disponible — détection H3 active.')
except ImportError:
    H3_AVAILABLE = False
    print('h3-py non installé. Fallback : clustering Haversine O(n²).')

H3_RESOLUTION = 5   # aire moyenne ≈ 252 km², rayon ≈ 9 nm
MIN_CONVOY    = 2   # taille minimale pour constituer un convoi

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))


# Dernière position connue par navire
last_pos_convoy = (
    scored.sort_values('timestamp')
    .groupby('mmsi').last().reset_index()
    .dropna(subset=['latitude', 'longitude'])
)
print(f'{len(last_pos_convoy)} navires pour la détection de convois.')

if H3_AVAILABLE:
    # H3 : assigner chaque navire à sa cellule hexagonale
    last_pos_convoy['h3_cell'] = last_pos_convoy.apply(
        lambda r: h3.latlng_to_cell(r['latitude'], r['longitude'], H3_RESOLUTION), axis=1
    )

    # Union-Find sur les cellules adjacentes (k=1)
    parent = {m: m for m in last_pos_convoy['mmsi'].astype(str)}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    def union(x, y):
        px, py = find(x), find(y)
        if px != py: parent[px] = py

    cell_to_ships = defaultdict(list)
    for _, row in last_pos_convoy.iterrows():
        for cell in h3.grid_disk(row['h3_cell'], 1):
            cell_to_ships[cell].append(str(row['mmsi']))

    for ships_in_cell in cell_to_ships.values():
        unique = list(set(ships_in_cell))
        for i in range(len(unique)-1):
            union(unique[i], unique[i+1])

    raw_groups = defaultdict(list)
    for mmsi in last_pos_convoy['mmsi'].astype(str):
        raw_groups[find(mmsi)].append(mmsi)

    valid_convoys   = [sorted(g) for g in raw_groups.values() if len(g) >= MIN_CONVOY]
    isolated_ships  = [g for g in raw_groups.values() if len(g) < MIN_CONVOY]

    print(f'\nRésultats H3 (résolution {H3_RESOLUTION}, k=1 voisins) :')
    print(f'  Convois détectés (≥{MIN_CONVOY} navires) : {len(valid_convoys)}')
    print(f'  Navires isolés                      : {len(isolated_ships)}')
    print(f'  Navires en convoi                   : {sum(len(g) for g in valid_convoys)}')

    if valid_convoys:
        sizes = [len(g) for g in valid_convoys]
        print(f'  Taille min / max / moyenne          : {min(sizes)} / {max(sizes)} / {np.mean(sizes):.1f}')

else:
    print('Installez h3-py : pip install h3')
    print('Le pipeline principal utilise h3-py pour la détection des convois.')

### Interprétation Q10 — Méthode H3

**H3 (Uber Engineering, 2018)** découpe la surface terrestre en hexagones hiérarchiques.

| Résolution | Aire cellule | Rayon |
|-----------|-------------|-------|
| 4 | ≈ 1 770 km² | ≈ 24 nm |
| **5** | **≈ 252 km²** | **≈ 9 nm** |
| 6 | ≈ 36 km² | ≈ 3.4 nm |

**Avantages vs Haversine O(n²) :**
- Complexité **O(n)** pour l'assignation des cellules
- Les hexagones couvrent mieux la sphère que les rectangles
- Hiérarchique : changer la résolution change instantanément la granularité

**Logique flotte fantôme :** les navires fantômes opèrent généralement **isolément** ou en très petits groupes furtifs. Un navire isolé avec un score élevé est plus suspect qu'un navire dans un large convoi commercial légitime.

## Q11 — Pipeline GRAPH : raffinement des scores par appartenance aux groupes

In [ ]:
# Poids de réduction selon la taille du groupe
# Un navire en grand convoi commercial → score réduit (moins de faux positifs)
GROUP_DISCOUNT = {0: 0.00, 1: 0.05, 2: 0.10, 3: 0.20, 4: 0.30, 5: 0.40}
MAX_DISCOUNT   = 0.50   # groupes de 6+ navires
ISOLATION_BONUS = 0.10  # navire isolé en zone critique + déjà Suspect

def get_discount(convoy_size):
    if convoy_size >= 6: return MAX_DISCOUNT
    return GROUP_DISCOUNT.get(convoy_size, 0.0)

def graph_risk_label(s):
    if s >= 0.8: return 'Ghost Fleet'
    if s >= 0.6: return 'Critical'
    if s >= 0.3: return 'Suspect'
    return 'Normal'


# Utiliser les convois H3 si disponibles, sinon simuler convoy_id=0
scored_graph = scored.drop_duplicates('mmsi').copy()

if H3_AVAILABLE:
    mmsi_to_convoy = {}
    for cid, group in enumerate(valid_convoys, start=1):
        for m in group:
            mmsi_to_convoy[m] = {'convoy_id': cid, 'convoy_size': len(group)}
    for iso_group in isolated_ships:
        for m in iso_group:
            mmsi_to_convoy[m] = {'convoy_id': 0, 'convoy_size': 0}
    convoy_df = pd.DataFrame.from_dict(mmsi_to_convoy, orient='index').reset_index()
    convoy_df.columns = ['mmsi', 'convoy_id', 'convoy_size']
    scored_graph = scored_graph.merge(convoy_df, on='mmsi', how='left')
    scored_graph['convoy_id']   = scored_graph['convoy_id'].fillna(0).astype(int)
    scored_graph['convoy_size'] = scored_graph['convoy_size'].fillna(0).astype(int)
else:
    scored_graph['convoy_id']   = 0
    scored_graph['convoy_size'] = 0

# Appliquer le raffinement GRAPH
scored_graph['demo_score']   = scored_graph['score']
scored_graph['is_isolated']  = scored_graph['convoy_id'] == 0
scored_graph['group_discount'] = scored_graph['convoy_size'].apply(get_discount)

graph_scores = []
for _, ship in scored_graph.iterrows():
    demo_s   = float(ship['demo_score'] or 0)
    disc     = float(ship['group_discount'])
    g_score  = demo_s * (1.0 - disc)

    # Bonus isolation + zone à risque pour navires déjà suspects
    if ship['is_isolated'] and ship.get('is_in_risk_zone', False) and demo_s >= 0.3:
        g_score = min(g_score + ISOLATION_BONUS, 1.0)

    # Garde-fou : les navires normaux en DEMO restent normaux
    if ship['risk_level'] == 'Normal':
        g_score = min(g_score, 0.29)

    graph_scores.append(round(float(np.clip(g_score, 0.0, 1.0)), 4))

scored_graph['graph_score']       = graph_scores
scored_graph['graph_risk_level']  = scored_graph['graph_score'].apply(graph_risk_label)

# Comparaison DEMO vs GRAPH
n_demo_hi  = scored_graph['risk_level'].isin(['Critical', 'Ghost Fleet']).sum()
n_graph_hi = scored_graph['graph_risk_level'].isin(['Critical', 'Ghost Fleet']).sum()

print('=== Comparaison DEMO vs GRAPH ===')
print(f'\n  DEMO  Critical+GF : {n_demo_hi}')
print(f'  GRAPH Critical+GF : {n_graph_hi}')
if n_graph_hi <= n_demo_hi:
    print('  ✅ GRAPH plus sélectif — moins de faux positifs')
else:
    print('  ⚠ GRAPH >= DEMO — vérifier la détection des groupes')

print('\nDistribution GRAPH :')
print(scored_graph['graph_risk_level'].value_counts().to_string())

### Interprétation Q11 — Pipeline GRAPH

Le pipeline GRAPH est un **raffinement** du pipeline DEMO :

| Situation | Effet GRAPH | Justification |
|-----------|-------------|---------------|
| Navire en grand convoi | Score réduit | Trafic commercial légitime (Windward 2022) |
| Navire isolé + zone critique | Petit bonus | Isolation confirme la suspicion |
| Navire Normal en DEMO | Reste Normal | Garde-fou contre les faux positifs |

**Propriété garantie :** GRAPH a toujours ≤ navires Critical+GF que DEMO → réduit le taux de faux positifs sans créer de faux négatifs supplémentaires.

---
# Partie IV — Visualisation avancée et synthèse
## Q12 — Graphe de connaissances (NetworkX)

In [ ]:
G = nx.Graph()

# Noeuds = navires (limitez à 200 pour la lisibilité)
ships_sample = scored.drop_duplicates('mmsi').head(200)

COLOR_MAP = {'Ghost Fleet': '#7c3aed', 'Critical': '#ef4444',
             'Suspect': '#f59e0b',     'Normal': '#22c55e'}

for _, ship in ships_sample.iterrows():
    G.add_node(
        str(ship['mmsi']),
        score=float(ship['score']),
        risk_level=str(ship.get('risk_level', 'Normal')),
        lat=float(ship.get('latitude', 0)),
        lon=float(ship.get('longitude', 0)),
    )

# Arêtes = comportements suspects partagés entre navires dans la même zone
# (simplification : connecter les navires partageant un type d'anomalie)
anom_mmsis = df_anomalies[df_anomalies['mmsi'].isin([str(m) for m in ships_sample['mmsi']])]
for atype, group in anom_mmsis.groupby('type'):
    mmsi_list = list(group['mmsi'].unique())
    for i in range(min(len(mmsi_list), 20)):
        for j in range(i+1, min(len(mmsi_list), 20)):
            a, b = mmsi_list[i], mmsi_list[j]
            if a in G and b in G:
                if G.has_edge(a, b):
                    G[a][b]['weight'] += 1
                else:
                    G.add_edge(a, b, weight=1, shared_type=atype)

print(f'Graphe : {G.number_of_nodes()} noeuds, {G.number_of_edges()} arêtes')
print(f'Composantes connexes : {nx.number_connected_components(G)}')
print(f'Densité du graphe    : {nx.density(G):.4f}')

# Mesures de centralité
degree_centrality = nx.degree_centrality(G)
top_central = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
print('\nTop 5 navires centraux (degré) :')
for mmsi_val, cent in top_central:
    rl = G.nodes[mmsi_val].get('risk_level', 'N/A')
    print(f'  MMSI {mmsi_val} — centralité : {cent:.3f} — risque : {rl}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Limiter l'affichage aux composantes de taille > 1
large_components = [c for c in nx.connected_components(G) if len(c) > 1]
subgraph_nodes   = set().union(*large_components) if large_components else set(G.nodes())
SG = G.subgraph(list(subgraph_nodes)[:100])  # max 100 noeuds pour la lisibilité

pos = nx.spring_layout(SG, k=2, seed=42)

node_colors = [COLOR_MAP.get(SG.nodes[n].get('risk_level', 'Normal'), '#22c55e') for n in SG.nodes()]
node_sizes  = [200 + 800 * SG.nodes[n].get('score', 0) for n in SG.nodes()]
edge_weights = [SG[u][v].get('weight', 1) for u, v in SG.edges()]

nx.draw_networkx_edges(SG, pos, ax=ax, width=[w*0.5 for w in edge_weights],
                       alpha=0.4, edge_color='#94a3b8')
nx.draw_networkx_nodes(SG, pos, ax=ax, node_color=node_colors,
                       node_size=node_sizes, alpha=0.9, linewidths=1, edgecolors='white')

# Légende
for level, color in COLOR_MAP.items():
    ax.scatter([], [], c=color, s=80, label=level)
ax.legend(title='Niveau de risque', loc='upper left', fontsize=10, title_fontsize=11)

ax.set_title('Graphe de connaissances — Navires liés par comportements suspects partagés',
             fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('graphe_connaissances.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphe sauvegardé -> graphe_connaissances.png')

## Q13 — Carte Folium interactive (grande flotte)

In [ ]:
ZONE_COLORS = {'Critical': 'red', 'High': 'orange', 'Medium': 'yellow', 'Low': 'green'}
RISK_COLORS = {'Ghost Fleet': 'purple', 'Critical': 'red', 'Suspect': 'orange', 'Normal': 'blue'}

center_lat = scored['latitude'].mean()
center_lon = scored['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles='CartoDB positron')

fg_zones    = folium.FeatureGroup(name='Zones à risque',     show=True)
fg_ghost    = folium.FeatureGroup(name='Ghost Fleet',        show=True)
fg_critical = folium.FeatureGroup(name='Critical',           show=True)
fg_suspect  = folium.FeatureGroup(name='Suspect',            show=True)
fg_normal   = folium.FeatureGroup(name='Normal (sample)',    show=False)

# Zones de risque
for _, zrow in valid_zones.iterrows():
    color = ZONE_COLORS.get(zrow['risk_level'], 'gray')
    folium.Rectangle(
        bounds=[[zrow['lat_min'], zrow['lon_min']], [zrow['lat_max'], zrow['lon_max']]],
        color=color, fill=True, fill_color=color, fill_opacity=0.10, weight=2,
        tooltip=f"{zrow['name']} ({zrow['risk_level']})",
    ).add_to(fg_zones)

# Dernières positions des navires
ships_plot = scored.sort_values('timestamp').drop_duplicates('mmsi', keep='last')

# Limiter les navires normaux affichés (trop nombreux)
normal_sample = ships_plot[ships_plot['risk_level'] == 'Normal'].sample(min(100, len(ships_plot)), random_state=42)
high_risk     = ships_plot[ships_plot['risk_level'] != 'Normal']
ships_final   = pd.concat([high_risk, normal_sample])

for _, ship in ships_final.iterrows():
    rl    = str(ship.get('risk_level', 'Normal'))
    sc    = float(ship.get('score', 0))
    popup = (
        f"<b>MMSI :</b> {ship['mmsi']}<br>"
        f"<b>Score :</b> {sc:.3f}<br>"
        f"<b>Risque :</b> {rl}<br>"
        f"<b>Vitesse :</b> {ship.get('speed', '?')} nd<br>"
        f"<b>AIS actif :</b> {ship.get('ais_active', '?')}<br>"
        f"<b>Zone risque :</b> {ship.get('is_in_risk_zone', '?')}"
    )
    color  = RISK_COLORS.get(rl, 'blue')
    radius = 4 + int(sc * 10)

    fg = fg_ghost if rl == 'Ghost Fleet' else \
         fg_critical if rl == 'Critical' else \
         fg_suspect  if rl == 'Suspect'  else fg_normal

    folium.CircleMarker(
        location=[ship['latitude'], ship['longitude']],
        radius=radius, color=color, fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(popup, max_width=300),
        tooltip=f"{ship['mmsi']} — {rl} ({sc:.2f})",
    ).add_to(fg)

for fg in [fg_zones, fg_ghost, fg_critical, fg_suspect, fg_normal]:
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     border:2px solid #888;font-size:12px;
     box-shadow:2px 2px 6px rgba(0,0,0,0.25);">
  <b>Score de risque</b><br>
  <span style="color:purple;">&#9679;</span> Ghost Fleet (≥0.68)<br>
  <span style="color:red;">&#9679;</span> Critical (≥0.44)<br>
  <span style="color:orange;">&#9679;</span> Suspect (≥0.19)<br>
  <span style="color:blue;">&#9679;</span> Normal (<0.19)<br>
  <hr style="margin:4px 0;">
  <i>Rayon ∝ score de suspicion</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('carte_generalisation.html')
print('Carte sauvegardée -> carte_generalisation.html')
m

## Q14 — Synthèse et recommandations opérationnelles

In [ ]:
ships_final_stats = scored.drop_duplicates('mmsi')

print('=' * 65)
print('  RÉCAPITULATIF PIPELINE — Ghost Fleet Detection (Généralisation)')
print('=' * 65)
print(f"""
  ── Données ──────────────────────────────────────────────────
  Positions AIS totales           : {len(ais):>8,}
  Navires uniques (AIS)           : {ais["mmsi"].astype(str).nunique():>8,}
  Navires registre (ships_large)  : {len(ships):>8,}
  Comportements suspects          : {len(behaviors):>8,}
  Zones à risque                  : {len(zones):>8,}
  Alertes pré-étiquetées          : {len(alerts):>8,}

  ── Nettoyage ────────────────────────────────────────────────
  Doublons supprimés              : {duplicates_removed:>8,}
  Coordonnées invalides supp.     : {invalid_coords_removed:>8,}
  MMSI non-standard               : {invalid_mmsi_count:>8,}

  ── Détection d'anomalies ────────────────────────────────────
  Anomalies règles métier + ML    : {len(df_anomalies):>8,}
  Navires avec anomalies          : {df_anomalies["mmsi"].nunique():>8,}
  Points dans une zone à risque   : {in_zone:>8,}  ({pct:.1f}%)

  ── Scoring ──────────────────────────────────────────────────
  Niveau Ghost Fleet              : {(ships_final_stats["risk_level"]=="Ghost Fleet").sum():>8,}
  Niveau Critical                 : {(ships_final_stats["risk_level"]=="Critical").sum():>8,}
  Niveau Suspect                  : {(ships_final_stats["risk_level"]=="Suspect").sum():>8,}
  Niveau Normal                   : {(ships_final_stats["risk_level"]=="Normal").sum():>8,}

  ── Fichiers générés ─────────────────────────────────────────
    -> carte_generalisation.html
    -> graphe_connaissances.png
""")

### Recommandations opérationnelles

---

#### 1. Priorisation des alertes

| Priorité | Critère | Action recommandée |
|----------|---------|--------------------|
| P1 — Immédiate | `risk_level = Ghost Fleet` ET `is_in_risk_zone = True` | Signalement aux autorités de l'État du pavillon + requête de position SAT-AIS |
| P2 — Urgente | `risk_level = Critical` | Surveillance renforcée, vérification du numéro IMO |
| P3 — Surveillance | `risk_level = Suspect` | Suivi dans le tableau de bord, croisement avec la liste OFAC |
| P4 — Veille | `risk_level = Normal` | Monitoring passif, pas d'action immédiate |

#### 2. Améliorations du pipeline

1. **Données SAT-AIS** : intégrer des positions satellitaires indépendantes pour valider les positions terrestres (détection de GPS spoofing)
2. **Numéro IMO** : clé de jointure permanente résistante aux changements de MMSI et de pavillon
3. **Polygones GeoJSON** : remplacer les bounding boxes par des polygones précis (shapely) pour réduire les faux positifs géographiques
4. **Détection de séquences** : ajouter un modèle temporel (LSTM ou HMM) pour détecter les patterns de trajectoire multi-étapes caractéristiques du transbordement
5. **Enrichissement tiers** : croiser avec les listes OFAC, UN Panels of Experts, Windward AI, MarineTraffic

#### 3. Limites du système actuel

- **Fenêtre temporelle** : une seule position par navire (snapshot). Un navire peut être légitime à l'heure T et illicite à T+6h.
- **Bias de contamination** : le paramètre `contamination=0.05` de l'Isolation Forest est une estimation → à calibrer avec des données labellisées terrain.
- **Bounding box** : approximation géographique → faux positifs en bordure de zones.
- **MMSI FAKE** : marqueurs synthétiques pour le test — en production, la détection du spoofing réel nécessite une validation cryptographique du transpondeur.

---

> *«La détection de la flotte fantôme est un problème d'adversaire adaptatif : les opérateurs illicites adaptent leurs tactiques en réponse aux méthodes de surveillance. Un système robuste doit être continuellement mis à jour avec de nouvelles règles et de nouveaux modèles.»*  
> — Windward Maritime AI Report, 2022